In [3]:
import pypsa
import pandas as pd
import numpy as np

In [ ]:
# Load the PyPSA network from the given NetCDF file (2035 scenario)
n = pypsa.Network('/Users/mick/Documents/GitHub/masterthesis-mick/base_s_1__none_2035_lt.nc')

# Extract time series of capacity factors for fixed-tilt solar in DE0 0
solar_cf = n.generators_t.p_max_pu['DE0 0 solar-2035']

# Extract time series of capacity factors for HSAT (single-axis tracking) solar in DE0 0
hsat_cf = n.generators_t.p_max_pu['DE0 0 solar-hsat-2035']

# Compute pointwise difference: HSAT minus fixed-tilt solar (positive => HSAT higher)
diff = hsat_cf.sub(solar_cf)

# Ensure the index is a pandas DateTimeIndex for time-based grouping
# get index to datetime
diff.index = pd.to_datetime(diff.index)

# Create a block identifier where each block spans 6 days since the series start
# block id to identify every block
block_id = ((diff.index - diff.index.min()).days // 6)

# For each (block, hour-of-day) pair, compute the mean difference
# for every block get the hourly mean
block_means = diff.groupby([block_id, diff.index.hour]).mean()

# Map the block/hour mean back to each original timestamp to form a full time series
# back to the original index for simpler application
diff_means = pd.Series(
    [block_means.loc[(b, h)] for b, h in zip(block_id, diff.index.hour)],
    index=diff.index
) 

In [7]:
pd.to_pickle(diff_means, "/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/cappacity_factors_prepared/diff_solar_to_solar-hsat.pkl")